# PT-W3-D6 概念实验：三套 ADR 在 Ontology 层统一

MI Domain Model 是世界模型层，CRE BCM ADR 是语义治理层，LangChat ADR 是执行层。统一的是共享语义锚点，不是把文档合并。

## 实验 1：Crosswalk 对齐与缺口登记

用 Capability、Entity Ownership、Relationship、Policy、Digital Employee 五个锚点，检查三层是否有一个权威来源或已显式登记缺口。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['axes.unicode_minus'] = False

CROSSWALK = {
    'Capability': {'MI': 'Capability Crosswalk: CRE-LEA-001', 'BCM': 'capability 行 CRE-LEA-001', 'LangChat': 'ApplicationContract: query_space'},
    'EntityOwnership': {'MI': 'Asset Foundation owns Space', 'BCM': 'ADR-003 Ownership Model', 'LangChat': 'Context scope: Lease Agent'},
    'Relationship': {'MI': 'Lease references Space', 'BCM': 'ADR-006 effect_type=occupancy', 'LangChat': 'Connector topology'},
    'Policy': {'MI': 'termination approval flow', 'BCM': 'effect governance', 'LangChat': 'conditional_write + scopes'},
    'DigitalEmployee': {'MI': None, 'BCM': None, 'LangChat': 'DigitalEmployeeDefinition'},
}
print('待统一的语义锚点:', list(CROSSWALK))

In [ ]:
def validate_anchor(name, layers):
    present = {layer: value for layer, value in layers.items() if value}
    missing = [layer for layer, value in layers.items() if not value]
    if name == 'DigitalEmployee' and missing == ['MI', 'BCM']:
        return name, 'registered_gap', 'MI/BCM 侧尚无锚点，作为 W4 显式缺口'
    if len(present) == 3:
        return name, 'aligned', '三层均有受控映射'
    return name, 'misaligned', f'缺少: {missing}'

checks = [validate_anchor(k, v) for k, v in CROSSWALK.items()]
for check in checks: print(check)
assert all(status in {'aligned', 'registered_gap'} for _, status, _ in checks)

In [ ]:
labels = ['对齐', '已登记缺口', '未对齐']
counts = [sum(s == 'aligned' for _, s, _ in checks), sum(s == 'registered_gap' for _, s, _ in checks), sum(s == 'misaligned' for _, s, _ in checks)]
coverage = np.array([[int(bool(CROSSWALK[a][layer])) for layer in ['MI', 'BCM', 'LangChat']] for a in CROSSWALK])
print('三层覆盖矩阵（行=锚点，列=MI/BCM/LangChat）', coverage)
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(labels, counts, color=['#2a9d8f', '#e9c46a', '#e76f51'])
ax.set_title('三套 ADR 的 Ontology 对齐验证'); ax.set_ylabel('锚点数')
plt.tight_layout(); plt.show()